In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [41]:
class SelfAttention(nn.Module):
    def __init__(self, emb_dim, dropout):
        super().__init__()
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)

        self.out_proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask: bool = True):
        batch_size, seq_len, emb_dim = x.size()

        q = self.query(x)  # (B, T, D)
        k = self.key(x)    # (B, T, D)
        v = self.value(x)  # (B, T, D)

        # Attention scores
        scores = q @ k.transpose(-2, -1) / (emb_dim ** 0.5)  # (B, T, T)

        if mask:
            causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).unsqueeze(0)  # (1, T, T)
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)  # (B, T, T)
        attn = self.dropout(attn)

        out = attn @ v  # (B, T, D)
        return self.out_proj(out)  # (B, T, D)

In [43]:
selfatt = SelfAttention(emb_dim=512,dropout=0.1)
x = torch.rand(2, 10, 512)
out = selfatt(x)
out.shape

torch.Size([2, 10, 512])

In [ ]:
mha = CrossAttention(emb_dim=512, num_heads=8, dropout=0.1)
decoder_hidden = torch.randn(2, 10, 512)  # Q
encoder_output = torch.randn(2, 20, 512)  # K, V
out = mha(decoder_hidden, context=encoder_output)
out.shape

In [ ]:
# need to add the length for catch
class KVCacheAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.register_buffer("cache_k", None)
        self.register_buffer("cache_v", None)

    def forward(self, x, use_cache=False):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        if use_cache:
            if self.cache_k is not None and self.cache_v is not None:
                k = torch.cat([self.cache_k, k], dim=1)
                v = torch.cat([self.cache_v, v], dim=1)
            self.cache_k = k
            self.cache_v = v

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (x.size(-1) ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)
        return torch.matmul(attn_weights, v)

In [ ]:
def precompute_theta_pos_freq(head_dim,seq_len,device,theta=10000.00):
    assert head_dim%2==0
    theta_numerator = torch.arange(0,head_dim,2).float()
    theta = 1.0 / (theta** (theta_numerator / head_dim)).to(device)
    m = torch.arange(seq_len,device=device)
    freq = torch.outer(m,theta).float()
    freq_complex = torch.polar(torch.ones_like(freq),freq)
    return freq_complex

def applay_rotry_pos_emb(x,freq_complex,device):
    
    x_complex = torch.view_as_complex(x.float().reshape(x.shape[-1],-1,2))
    freq_complex = freq_complex.unsqueeze(0).unsqueeze(2)
    x_rotated = x_complex * freq_complex
    x_out = x_out.reshape(x.shape)
    return x.out.type_as(x).to(device)

def repeat_kv(x,n_repeats):
    batch_size,seq_len,n_kv_heads,head_dim = x.size
    
    if n_repeats == 1:
        return x
    
    return (
        # batch, seq_len, n_kv_heads, 1, head_dim
        x[:,:,:,None,:]
        # batch, seq_len, n_kv_heads, n_rep, heaf_dim
        .extend(batch_size,seq_len,n_kv_heads,n_repeats,head_dim)
        # batch, seq_len, n_kv_heads * n_repe, head_dim
        .reshape(batch_size,seq_len,n_kv_heads*n_repeats,head_dim)
    )


class KVCacheAttention(nn.Module): # need to add the sequence length for key and value
    def __init__(self, embed_dim):
        super().__init__()
        
        self.n_kv_heads = n_heaf is None else n_kv_heads
        self.n_q_heads = args_q_heads
        self.n_reap = self.n_q_heads // self.n_kv_heads
        self.head_dim = dim // n_heads
        
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim,embed_dim,bias=False)
        
        self.cache_keys = torch.zeros((max_batch_size,max_seq_len,n_kv_heads,head_dim))
        self.cache_values = torch.zeros((max_batch_size,max_seq_len,n_kv_heads,head_dim))
        
    def forward(self, x, start_pos, freq):
        Batch_size, Seq_len, emd_dim = x.shape
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        
        query = q.view(Batch_size, Seq_len,nq_heads,head_dim)
        key = k.view(Batch_size, Seq_len,nk_heads,head_dim)
        value = v.view(Batch_size, Seq_len,nv_heads,head_dim)
        
        query = applay_rotry_pos_emb(query,start_pos,device = x.device)
        key = applay_rotry_pos_emb(key,start_pos,device = x.device)
        
        # entry replace in catch
        self.cache_keys[:Batch_size,start_pos:start_pos:seq_len] = key
        self.cache_value[:Batch_size,start_pos:start_pos:seq_len] = value
        
        # retrive catch so far
        keys = self.cache_keys[:Batch_size,0:start_pos+seq_len]
        values = self.cache_value[:Batch_size,0:start_pos+seq_len]
        
        # repeat kv
        keys = repeat_kv(keys,self.n_rep)
        values = repeat_kv(values,self.n_rep)
        
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (x.size(-1) ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)
        return torch.matmul(attn_weights, v)


In [ ]:
# check and verify those thisgs and see about the masking
class flash_attention_Head(nn.Module):
    def __init__(self, emd_dim, block_size, head_size,Dropout):
        super().__init__()
        self.key = nn.Linear(emd_dim, head_size, bias=False)
        self.query = nn.Linear(emd_dim, head_size, bias=False)
        self.value = nn.Linear(emd_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(Dropout)
        
    def forward(self, x):
        batch, seq_len, emd_dim = x.shape  # batch, seq_len, channels

        k = self.key(x)    # (Batch_size, Seq_len, head_size)
        q = self.query(x)  # (Batch_size, Seq_len, head_size)
        v = self.value(x)  # (Batch_size, Seq_len, head_size)

        output = torch.zeros_like(q)
        
        for b in range(batch):
            for i in range(seq_len):
                # 1. Compute q_i . k_j for all j (dot product row)
                q_i = q[b,i]
                attn_score = k[b] @ q_i
                # 2. Numerical stability: subtract max
                max_score = attn_score.max()
                stable_score = attn_score - max_score
                
                # 3. Exponentiate + normalize
                weights = torch.exp(stable_score)
                weights = weights / weights.sum()
                
                # 4. Weighted sum over V
                output[b,i] = weights @ v[b]
                
        return output

class FlashAttention(nn.Module):
    def __init__(self, num_heads, emd_dim, block_size, dropout):
        super().__init__()
        head_size = emd_dim // num_heads  # Ensure total heads combine to emd_dim
        self.heads = nn.ModuleList([flash_attention_Head(emd_dim, block_size, head_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emd_dim, emd_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out